In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:

df.drop_duplicates(inplace=True)

In [7]:
df.reset_index(drop=True,inplace=True)

In [8]:
df['sentiment'].value_counts()

sentiment
positive    24884
negative    24698
Name: count, dtype: int64

In [9]:
import re
df['review']=df['review'].str.lower()
def clean_text(text):
    text=re.sub(r"https\S+","",text)
    text=re.sub(r"<.*?>","",text)
    text=re.sub(r"[^a-zA-Z0-9\s]","",text)
    return text

df['review']=df['review'].apply(clean_text)


In [10]:
df['review'].iloc[1]

'a wonderful little production the filming technique is very unassuming very oldtimebbc fashion and gives a comforting and sometimes discomforting sense of realism to the entire piece the actors are extremely well chosen michael sheen not only has got all the polari but he has all the voices down pat too you can truly see the seamless editing guided by the references to williams diary entries not only is it well worth the watching but it is a terrificly written and performed piece a masterful production about one of the great masters of comedy and his life the realism really comes home with the little things the fantasy of the guard which rather than use the traditional dream techniques remains solid then disappears it plays on our knowledge and our senses particularly with the scenes concerning orton and halliwell and the sets particularly of their flat with halliwells murals decorating every surface are terribly well done'

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer=Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df["review"])        #makes dictionary
sequences=tokenizer.texts_to_sequences(df['review'])   #now assign numbers to the words from dictionary
    

In [12]:
print(len(sequences[1]))

140


In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X=pad_sequences(sequences,maxlen=100)

In [14]:
X.shape

(49582, 100)

In [15]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['sentiment']=le.fit_transform(df['sentiment'])

In [16]:
y=df['sentiment']

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
print(X_train.shape)
print(X_test.shape)

(39665, 100)
(9917, 100)


In [19]:
import torch

X_train=torch.tensor(X_train,dtype=torch.long)
X_test=torch.tensor(X_test,dtype=torch.long)

y_train=torch.tensor(y_train.values,dtype=torch.float32)
y_test=torch.tensor(y_test.values,dtype=torch.float32)

In [20]:
print(X_train[0])
print(y_train[0])

tensor([   1,  164,   17,  144,    9,   12,   36,  556,  340,    2,  303,  121,
         325,  257, 4443,   14,    3,  630,    9,   16,   12,   80,   28,   36,
        1374,  100, 1569,   10,  225,  418,    5,  885,  296,  406,   17,   10,
          88,   52,  116,   46,  326,    5,  124,   10, 1378,    3,  355,    4,
         211,   17,   60,   81,   10,   12, 2760,    3,   16,   36,   11,  138,
          29,  214,   25,  154,   51,  121, 1981,    8,   12,   20,  275,   22,
          39,  154,    2, 1914, 1054,   83,  681,  352,   96,   39, 1228,   14,
           1,  164,   44,   21,  173,    3,  154, 1054,   16,  359,  296,    6,
           9,  359,  204,  133])
tensor(0.)


In [21]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self,vocab_size,embed_size=128,hidden_size=128):
        super(LSTMModel,self).__init__()

        self.embedding=nn.Embedding(vocab_size,embed_size)

        self.lstm=nn.LSTM(embed_size,hidden_size,batch_first=True)

        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        x=self.embedding(x)
        h0=torch.zeros(1,x.size(0),128)
        c0=torch.zeros(1,x.size(0),128)
        out,_=self.lstm(x,(h0,c0))
        out=out[:,-1,:]
        out=self.fc(out)
        return out

In [22]:
from torch.utils.data import TensorDataset,DataLoader

train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)

train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64)


In [23]:
model=LSTMModel(vocab_size=5001)

In [24]:
import torch.optim as optim
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.01)

In [25]:
epochs=5

for epoch in range(epochs):
    model.train()
    total_loss=0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs=model(xb)

        outputs=torch.sigmoid(outputs.squeeze())

        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.4362237026133845
Epoch 2, Loss: 0.3039462736176868
Epoch 3, Loss: 0.26538855581273957
Epoch 4, Loss: 0.2379399751102732
Epoch 5, Loss: 0.22254863384990922


In [26]:
model.eval()
correct=0
total=0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs=model(xb)
        preds=(torch.sigmoid(outputs.squeeze())>0.5).float()

        correct+=(preds==yb).sum().item()
        total+=yb.size(0)
accuracy=correct/total
print("Accuracy: ",accuracy*100)


Accuracy:  84.60219824543714


In [27]:
def predict_sentiment(text):
    text = clean_text(text)

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=100)

    tensor = torch.tensor(padded, dtype=torch.long)

    output = model(tensor)
    prob = torch.sigmoid(output).item()

    return "Positive " if prob > 0.5 else "Negative "

In [28]:
print(predict_sentiment("The plot was confusing but acting was great"))


Positive 


In [29]:
torch.save(model.state_dict(), "LSTMmodel.pth")

In [30]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)